# Day 021 Project: AI File Organizer

## What You're Building

An end-to-end AI pipeline that scans a directory, reads every file, asks a local LLM to categorize each one, and produces two outputs:

- `manifest.json` — full metadata for every file (name, path, category, tags, summary)
- `report.csv` — a tidy summary table (name, category, summary, size_bytes)

You also organize the files into category subfolders.

**You run it, it produces those two files and a sorted folder. That's the deliverable.**

In [ ]:
import json
import csv
import shutil
import tempfile
from pathlib import Path
import ollama

## Provided: All Helper Functions

In [ ]:
def scan_directory(directory: str, pattern: str = "*") -> list[dict]:
    result = []
    for item in Path(directory).glob(pattern):
        if item.is_file():
            result.append({
                "name": item.name,
                "path": str(item),
                "size_bytes": item.stat().st_size,
                "extension": item.suffix,
            })
    return result


def read_csv(path: str) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: str, rows: list[dict], fieldnames: list[str]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def load_json_files(directory: str) -> list[dict]:
    results = []
    for p in Path(directory).glob("*.json"):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(data, dict):
                data["_source"] = p.name
                results.append(data)
        except Exception:
            pass
    return results


def batch_process_files(directory: str, process_fn) -> list[dict]:
    results = []
    for p in sorted(Path(directory).glob("*")):
        if not p.is_file():
            continue
        try:
            content = p.read_text(encoding="utf-8")
            result = process_fn(content)
            results.append({"path": str(p), "status": "ok", "result": result})
        except Exception as e:
            results.append({"path": str(p), "status": "error", "error": str(e)})
    return results


def ai_tag_file(content: str, model: str = "llama3.2") -> dict:
    system = (
        "You are a file categorization assistant. "
        "Given text content, return JSON with exactly these keys: "
        "category (one word: technical, personal, financial, creative, or other), "
        "tags (list of up to 5 keyword strings), "
        "summary (one sentence describing the content). "
        "Return only valid JSON."
    )
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": f"Categorize this content:\n\n{content[:2000]}"},
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        data = json.loads(raw)
    except Exception:
        data = {}
    return {
        "category": str(data.get("category", "other")).lower().strip() or "other",
        "tags": list(data.get("tags", [])),
        "summary": str(data.get("summary", "")),
    }

## Your Implementation

### Step 1: Create your sample corpus

Define at least **5 diverse text files** as a dict, write them to a temp directory, then scan it.

In [ ]:
# Create sample corpus — add at least 5 files covering different topics
SAMPLE_FILES = {
    'python_tutorial.txt': (
        'Python is a high-level programming language known for clean syntax. '
        'It supports procedural, object-oriented, and functional programming. '
        'Python is widely used in data science, web development, and automation.'
    ),
    'budget_2024.txt': (
        'Q1 Budget Summary: Revenue $45,000. Expenses: rent $12,000, '
        'salaries $18,000, software subscriptions $2,500, marketing $3,000. '
        'Net profit Q1: $9,500.'
    ),
    'poem_nature.txt': (
        'Autumn leaves descend / golden light on still water / silence finds me here. '
        'The river speaks low / carrying dreams to the sea / where time dissolves.'
    ),
    'ml_notes.txt': (
        'Neural networks learn by adjusting weights during backpropagation. '
        'The transformer architecture uses self-attention to model token relationships. '
        'Large language models are pre-trained on vast text corpora.'
    ),
    'shopping_list.txt': (
        'Weekly groceries: milk, eggs, whole wheat bread, cheddar cheese, '
        'chicken breast, broccoli, spinach, olive oil, pasta, tomato sauce.'
    ),
}

# Write files to a temp directory
SAMPLE_DIR = Path(tempfile.mkdtemp())
for fname, content in SAMPLE_FILES.items():
    (SAMPLE_DIR / fname).write_text(content, encoding='utf-8')
print(f'Created {len(SAMPLE_FILES)} sample files in {SAMPLE_DIR}')

### Step 2: Scan the directory

In [ ]:
# Scan and inspect the files
file_list = scan_directory(str(SAMPLE_DIR))
print(f'Found {len(file_list)} files:')
for f in file_list:
    print(f"  {f['name']} ({f['size_bytes']} bytes)")

### Step 3: AI-tag each file

In [ ]:
# Tag every file with the LLM
results = []
for file_info in file_list:
    content = Path(file_info['path']).read_text(encoding='utf-8')
    tags = ai_tag_file(content)
    results.append({**file_info, **tags})
    print(f"  {file_info['name']} -> {tags['category']}: {tags['summary'][:60]}...")

### Step 4: Write manifest.json

In [ ]:
MANIFEST_PATH = SAMPLE_DIR / 'manifest.json'
MANIFEST_PATH.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'Wrote {MANIFEST_PATH}')

### Step 5: Write report.csv

In [ ]:
REPORT_PATH = SAMPLE_DIR / 'report.csv'
write_csv(
    str(REPORT_PATH),
    results,
    fieldnames=['name', 'category', 'summary', 'size_bytes'],
)
report_rows = read_csv(str(REPORT_PATH))
print(f'Wrote {REPORT_PATH} ({len(report_rows)} rows)')

### Step 6: Organize files into category folders

In [ ]:
ORGANIZED_DIR = SAMPLE_DIR / 'organized'
ORGANIZED_DIR.mkdir()
for item in results:
    cat_dir = ORGANIZED_DIR / item['category']
    cat_dir.mkdir(exist_ok=True)
    shutil.copy(item['path'], cat_dir / item['name'])
print('Files organized:')
for cat_dir in sorted(ORGANIZED_DIR.iterdir()):
    if cat_dir.is_dir():
        print(f"  {cat_dir.name}/: {len(list(cat_dir.iterdir()))} file(s)")

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: SAMPLE_DIR has >= 3 files
    try:
        assert 'SAMPLE_DIR' in globals(), 'SAMPLE_DIR not defined'
        sample_path = Path(SAMPLE_DIR)
        assert sample_path.is_dir(), f'{SAMPLE_DIR} is not a directory'
        n_files = len([f for f in sample_path.glob('*') if f.is_file()])
        assert n_files >= 3, f'need >= 3 files, found {n_files}'
        passed += 1; print(f'\u2705 Check 1: SAMPLE_DIR has {n_files} files')
    except Exception as e:
        print(f'\u274c Check 1: SAMPLE_DIR — {e}')

    # Check 2: file_list is populated
    try:
        assert 'file_list' in globals(), 'file_list not defined'
        assert isinstance(file_list, list) and len(file_list) >= 3
        for item in file_list:
            assert 'name' in item and 'size_bytes' in item
        passed += 1; print(f'\u2705 Check 2: file_list has {len(file_list)} metadata dicts')
    except Exception as e:
        print(f'\u274c Check 2: file_list — {e}')

    # Check 3: results has category/tags/summary
    try:
        assert 'results' in globals(), 'results not defined'
        assert isinstance(results, list) and len(results) >= 3
        for r in results:
            for key in ('category', 'tags', 'summary'):
                assert key in r, f"missing '{key}' in: {r}"
        passed += 1; print(f'\u2705 Check 3: results has {len(results)} tagged items')
    except Exception as e:
        print(f'\u274c Check 3: results — {e}')

    # Check 4: MANIFEST_PATH exists and is valid JSON
    try:
        assert 'MANIFEST_PATH' in globals(), 'MANIFEST_PATH not defined'
        mp = Path(MANIFEST_PATH)
        assert mp.exists(), f'{MANIFEST_PATH} does not exist'
        json.loads(mp.read_text(encoding='utf-8'))
        passed += 1; print('\u2705 Check 4: manifest.json exists and is valid JSON')
    except Exception as e:
        print(f'\u274c Check 4: MANIFEST_PATH — {e}')

    # Check 5: REPORT_PATH exists with data rows
    try:
        assert 'REPORT_PATH' in globals(), 'REPORT_PATH not defined'
        rp = Path(REPORT_PATH)
        assert rp.exists(), f'{REPORT_PATH} does not exist'
        rows = read_csv(str(rp))
        assert len(rows) >= 1, 'report.csv has no data rows'
        passed += 1; print(f'\u2705 Check 5: report.csv exists with {len(rows)} rows')
    except Exception as e:
        print(f'\u274c Check 5: REPORT_PATH — {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Use `batch_process_files` to collect word counts before tagging — skip files with fewer than 20 words
- Persist results to `manifest.json` incrementally so a crash mid-run doesn't lose finished work
- Add a `--dry-run` mode: print what would be organized without moving files
- Try `load_json_files` on the `organized/` subfolders to re-aggregate results
- Extend `scan_directory` to recurse with `.rglob()` and add a `depth` counter